# Dataclasses in Python (with Trading Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to Python's `dataclasses` module.

You already know OOP; here we focus on **what dataclasses are, why they exist, and how to use them effectively** in real-world code, using **trading-related examples**:

- Orders and positions
- Market data ticks
- Risk and configuration models

We will build up gradually:

1. **Core ideas**: what `@dataclass` does and basic usage
2. **Fields, defaults, and `field()` factories**
3. **Post-processing with `__post_init__` (validation, derived fields)**
4. **Comparison, hashing, and immutability (`frozen=True`)**
5. **Inheritance, composition, and `slots=True`**
6. **Converting to/from dicts, copying, and updating (`asdict`, `replace`)**
7. **Trading case studies**: orders/positions, risk configs, and tick data
8. **Best practices, pitfalls, and when *not* to use dataclasses

> Run and modify the code cells as you go. The goal is to make dataclasses feel like a
> natural tool for modeling trading-domain data, not just a decorator you add by habit.

### Contents

- [1. Core Ideas: What Dataclasses Do](#1-core-ideas-what-dataclasses-do)
- [2. Fields, Defaults, and `field()`](#2-fields-defaults-and-field)
- [3. `__post_init__`: Validation and Derived Fields](#3-__post_init__-validation-and-derived-fields)
- [4. Comparison, Ordering, Hashing, and Immutability](#4-comparison-ordering-hashing-and-immutability)
- [5. Inheritance, Composition, and `slots=True`](#5-inheritance-composition-and-slotstrue)
- [6. `asdict`, `astuple`, and `replace`](#6-asdict-astuple-and-replace)
- [7. Trading Case Study: Orders, Positions, and Ticks with Dataclasses](#7-trading-case-study-orders-positions-and-ticks-with-dataclasses)
- [8. Best Practices, Pitfalls, and When Not to Use Dataclasses](#8-best-practices-pitfalls-and-when-not-to-use-dataclasses)

## 1. Core Ideas: What Dataclasses Do

The `dataclasses` module (Python 3.7+) provides a decorator, `@dataclass`, that
**writes boilerplate methods for you** based on type-annotated fields:

- `__init__`
- `__repr__`
- `__eq__` (and optionally ordering methods)
- Sometimes `__hash__`

The typical use case is a class that mainly **stores data** with a bit of logic.
This fits trading code nicely: orders, positions, ticks, configs, etc.

Let's start with a simple trading order example.

In [ ]:
# 1.1 A simple order without dataclasses

class Order:
    def __init__(self, symbol: str, side: str, quantity: float, limit_price: float | None = None):
        self.symbol = symbol
        self.side = side
        self.quantity = quantity
        self.limit_price = limit_price

    def __repr__(self) -> str:
        return (
            f"Order(symbol={self.symbol!r}, side={self.side!r}, "
            f"quantity={self.quantity!r}, limit_price={self.limit_price!r})"
        )


o1 = Order("AAPL", "BUY", 100)
o2 = Order("AAPL", "BUY", 100)

print(o1)
print("o1 == o2?", o1 == o2)  # This will be False: default object identity comparison

In [ ]:
# 1.2 The same order as a dataclass

from dataclasses import dataclass


@dataclass
class DCOrder:
    symbol: str
    side: str
    quantity: float
    limit_price: float | None = None


od1 = DCOrder("AAPL", "BUY", 100)
od2 = DCOrder("AAPL", "BUY", 100)

print(od1)  # Auto-generated __repr__
print("od1 == od2?", od1 == od2)  # Auto-generated __eq__ compares field values

### 1.3 What `@dataclass` generated for us

For `DCOrder`, `@dataclass` roughly generated:

- An `__init__(self, symbol, side, quantity, limit_price=None)` that assigns attributes.
- A value-based `__eq__` that compares all fields.
- A nice `__repr__` that prints field names and values.

This is **purely about boilerplate reduction** and **value semantics**.

Next, we look at **fields, defaults, and `field()`** to control behavior more precisely.

## 2. Fields, Defaults, and `field()`

Dataclass fields are defined by **type-annotated attributes** in the class body.

You can control each field using the `field()` helper:

- `default=...` / `default_factory=...`
- `init` (include in `__init__` or not)
- `repr` (include in `__repr__` or not)
- `compare` (include in `__eq__` / ordering or not)
- `metadata` (attach arbitrary metadata)

We'll build a **trading position** model that uses several of these options.

In [ ]:
# 2.1 Position with defaults and field() options

from dataclasses import dataclass, field
from datetime import datetime
from typing import Literal

Side = Literal["LONG", "SHORT"]


@dataclass
class Position:
    symbol: str
    side: Side
    quantity: float = 0.0
    avg_price: float = 0.0
    opened_at: datetime = field(default_factory=datetime.utcnow, repr=False)
    tags: list[str] = field(default_factory=list)

    def market_value(self, price: float) -> float:
        return self.quantity * price * (1 if self.side == "LONG" else -1)


p = Position("AAPL", side="LONG", quantity=100, avg_price=180.0)
print(p)
print("Market value @ 190:", p.market_value(190.0))
print("Opened at (hidden in repr):", p.opened_at)
print("Tags:", p.tags)


### 2.2 Notes on defaults and `default_factory`

- `quantity` and `avg_price` use simple scalar defaults.
- `opened_at` uses `default_factory=datetime.utcnow` so that each instance gets the
  **current time** when it is created.
- `tags` uses `default_factory=list` to avoid the classic **mutable default argument** trap.

Use `default_factory` whenever the default value should be **computed at runtime** or
be a **fresh container** (`list`, `dict`, etc.) for each instance.

Next, we use `__post_init__` to run validation and derive fields.

## 3. `__post_init__`: Validation and Derived Fields

Dataclasses support a special hook:

- `__post_init__(self)`: called **automatically after `__init__`**.

Typical uses:

- Validate field combinations (e.g. side/quantity consistency).
- Compute **derived fields** (e.g. signed quantity, notional).
- Normalize inputs (e.g. uppercasing symbols).

We'll model a **normalized order** with some simple checks.

In [ ]:
# 3.1 Order with __post_init__ validation and derived fields

from dataclasses import dataclass, field
from typing import Literal

SideLR = Literal["BUY", "SELL"]


@dataclass
class NormalizedOrder:
    symbol: str
    side: SideLR
    quantity: float
    limit_price: float | None = None
    signed_qty: float = field(init=False)

    def __post_init__(self) -> None:
        # Normalize symbol and side
        self.symbol = self.symbol.upper()
        self.side = self.side.upper()  # type: ignore[assignment]

        if self.side not in ("BUY", "SELL"):
            raise ValueError("side must be 'BUY' or 'SELL'")
        if self.quantity <= 0:
            raise ValueError("quantity must be positive")

        # Derived signed quantity (+ for buy, - for sell)
        self.signed_qty = self.quantity if self.side == "BUY" else -self.quantity


no = NormalizedOrder("aapl", "buy", 100)
print(no)
print("Signed quantity:", no.signed_qty)


### 3.2 Takeaways from `__post_init__`

- Use `field(init=False)` for attributes that are **derived**, not passed by callers.
- `__post_init__` gives you a single place to **enforce invariants**.
- This keeps your `__init__` clean and auto-generated while still allowing
  rich domain logic.

Next, we'll look at **comparison, hashing, and immutability**.

## 4. Comparison, Ordering, Hashing, and Immutability

By default, dataclasses generate `__eq__` and may generate `__hash__` depending on options.
You can control this via decorator arguments:

- `order=True`: generate ordering methods (`__lt__`, `__le__`, etc.).
- `frozen=True`: make instances **immutable** (fields cannot be reassigned).
- `unsafe_hash=True`: force generation of a hash even when it is normally unsafe.

In trading, you may want **immutable value objects** for things like **execution reports**
or **configuration snapshots**.

Let's build an immutable `ExecutionReport` that is orderable by time.

In [ ]:
# 4.1 Immutable, orderable execution report

from dataclasses import dataclass
from datetime import datetime
from typing import Literal

Status = Literal["NEW", "PARTIAL", "FILLED", "CANCELLED"]


@dataclass(order=True, frozen=True)
class ExecutionReport:
    timestamp: datetime
    order_id: int
    symbol: str
    status: Status
    filled_qty: float
    remaining_qty: float


now = datetime.utcnow()
reports = [
    ExecutionReport(now, 1, "AAPL", "NEW", 0, 100),
    ExecutionReport(now, 2, "AAPL", "FILLED", 100, 0),
    ExecutionReport(now, 3, "MSFT", "PARTIAL", 50, 50),
]

print("Sorted reports by timestamp, order_id:")
for r in sorted(reports):
    print(r)

# Try mutating (will raise FrozenInstanceError)
try:
    reports[0].status = "FILLED"  # type: ignore[misc]
except Exception as e:
    print("Cannot modify frozen report:", type(e).__name__, e)


### 4.2 When to use `frozen` and `order`

- **`frozen=True`**:
  - Use for **value objects** that should not change after creation.
  - Makes them hashable (if fields are hashable), so you can put them in `set`/`dict` keys.
- **`order=True`**:
  - Use when you have a **natural sort order** (e.g. by time, then id).
  - Ordering uses the field definition order as a tuple key.

Be careful with mutability: if you need to update frequently, a frozen dataclass
may be inconvenient; consider keeping them mutable but treat them immutably by convention.

## 5. Inheritance, Composition, and `slots=True`

Dataclasses work with **inheritance** and **composition**:

- Subclasses can add fields on top of base dataclasses.
- You can nest dataclasses inside other dataclasses.

Python 3.10+ also supports `slots=True` in `@dataclass` to reduce memory usage and
speed up attribute access (good for many small objects like **ticks** or **orders**).

We'll build a small hierarchy of instruments and orders using these features.

In [ ]:
# 5.1 Instrument + order hierarchy with composition and slots

from dataclasses import dataclass
from typing import Literal


@dataclass(slots=True)
class Instrument:
    symbol: str
    asset_class: str  # e.g. 'equity', 'future'


@dataclass(slots=True)
class EquityInstrument2(Instrument):
    tick_size: float = 0.01


Side2 = Literal["BUY", "SELL"]


@dataclass(slots=True)
class BaseOrderDC:
    instrument: Instrument
    side: Side2
    quantity: float


@dataclass(slots=True)
class MarketOrderDC(BaseOrderDC):
    pass


@dataclass(slots=True)
class LimitOrderDC(BaseOrderDC):
    limit_price: float


inst_eq = EquityInstrument2("AAPL", asset_class="equity", tick_size=0.01)
mo = MarketOrderDC(inst_eq, side="BUY", quantity=100)
lo = LimitOrderDC(inst_eq, side="SELL", quantity=50, limit_price=190.0)

print(mo)
print(lo)


### 5.2 Notes on inheritance and `slots`

- Subclass dataclasses must generally **also** be decorated with `@dataclass`.
- `slots=True` prevents adding new attributes at runtime and can significantly
  reduce memory footprint for many instances.
- Composition (`instrument` inside orders) keeps the model flexible: you can
  reuse the same order classes for different instrument types.

Next, we'll look at **converting dataclasses to/from dicts** and updating them.

## 6. `asdict`, `astuple`, and `replace`

Dataclasses come with helper functions:

- `dataclasses.asdict(obj)` → deep-convert to nested dicts.
- `dataclasses.astuple(obj)` → deep-convert to tuples.
- `dataclasses.replace(obj, **changes)` → copy with some fields changed.

These are handy for **logging, serialization, and immutable-style updates**.
We'll use them with a simple **risk configuration** dataclass.

In [ ]:
# 6.1 Working with asdict, astuple, and replace

from dataclasses import dataclass, asdict, astuple, replace


@dataclass
class SymbolRiskConfig:
    symbol: str
    max_position: float
    max_notional: float
    max_daily_trades: int = 0


cfg1 = SymbolRiskConfig("AAPL", max_position=1000, max_notional=100_000.0, max_daily_trades=10)

print("cfg1 as dict:", asdict(cfg1))
print("cfg1 as tuple:", astuple(cfg1))

# Create a modified copy with a different limit
cfg2 = replace(cfg1, max_position=500)
print("cfg2 (reduced position limit):", cfg2)

# Original unchanged
print("cfg1 still:", cfg1)


## 7. Trading Case Study: Orders, Positions, and Ticks with Dataclasses

Now we build a slightly more realistic mini-model using dataclasses:

- `Tick`: market data updates.
- `OrderRequest`: a request to send an order.
- `Fill`: execution information.
- `PositionState`: tracks position and PnL.

We will see how dataclasses make it easy to express **plain data structures** while
keeping domain logic close to the data.

In [ ]:
# 7.1 Mini trading model with dataclasses

from dataclasses import dataclass, field
from datetime import datetime
from typing import Literal

Side3 = Literal["BUY", "SELL"]


@dataclass(slots=True)
class Tick:
    symbol: str
    price: float
    timestamp: datetime


@dataclass(slots=True)
class OrderRequest:
    symbol: str
    side: Side3
    quantity: float
    limit_price: float | None = None
    created_at: datetime = field(default_factory=datetime.utcnow)


@dataclass(slots=True)
class Fill:
    symbol: str
    side: Side3
    quantity: float
    price: float
    timestamp: datetime


@dataclass
class PositionState:
    symbol: str
    quantity: float = 0.0
    avg_price: float = 0.0

    def apply_fill(self, fill: Fill) -> None:
        if fill.symbol != self.symbol:
            raise ValueError("Fill symbol does not match position symbol")

        signed_qty = fill.quantity if fill.side == "BUY" else -fill.quantity
        new_qty = self.quantity + signed_qty

        if self.quantity == 0 or (self.quantity > 0) == (signed_qty > 0):
            # Increasing in same direction → update average price
            total_cost = self.avg_price * abs(self.quantity) + fill.price * abs(signed_qty)
            total_qty = abs(self.quantity) + abs(signed_qty)
            self.avg_price = total_cost / total_qty
        # else: reducing or flipping → keep old avg_price (simplified)

        self.quantity = new_qty

    def unrealized_pnl(self, price: float) -> float:
        return (price - self.avg_price) * self.quantity


# Demo: simulate a simple trade lifecycle

pos = PositionState("AAPL")

fills = [
    Fill("AAPL", "BUY", 100, 180.0, datetime.utcnow()),
    Fill("AAPL", "SELL", 50, 185.0, datetime.utcnow()),
]

for f in fills:
    pos.apply_fill(f)

print("Position:", pos)
print("Unrealized PnL @ 190:", pos.unrealized_pnl(190.0))


### 7.2 Takeaways from the case study

- Dataclasses keep the code **compact and readable**, even with several related types.
- Business logic (`apply_fill`, `unrealized_pnl`) lives **next to the data it uses**.
- Using `slots=True` for frequently created types (`Tick`, `OrderRequest`, `Fill`) can
  improve memory usage when processing large data streams.

You can extend this model by adding more fields (e.g. `order_id`, `exchange`), or
by composing it with ABCs/metaclasses from the other notebooks.

## 8. Best Practices, Pitfalls, and When Not to Use Dataclasses

### 8.1 Good use cases

- **Data-heavy domains** with lightweight logic:
  - Orders, fills, positions, ticks, risk configs.
- When you want **value semantics** (`==` means same data) instead of identity.
- When you need easy **serialization** and **logging**.

### 8.2 When dataclasses are not ideal

- Very behavior-heavy classes with lots of methods and little data.
- When you need full control over `__init__` / `__new__` (e.g. singletons, flyweights).
- When your team finds the generated methods confusing and the manual boilerplate
  would actually be clearer.

### 8.3 Common pitfalls

- **Mutable default values**: always use `default_factory` for lists/dicts/sets.
- **Leaking implementation details**: don't automatically include every internal field
  in comparison/ordering if it doesn't make sense for business equality.
- **Overusing `frozen=True`**: immutability is great, but can make updates awkward if
  your domain model is naturally mutable.

### 8.4 How to practice

- Convert a few existing POJO-style classes in your own codebase into dataclasses.
- Add `__post_init__` to enforce invariants and normalize inputs.
- Introduce `asdict` / `replace` for logging and immutable-style updates.

Once these patterns feel natural, dataclasses become a very ergonomic way to model
trading data structures and glue them into the ABCs/metaclasses patterns from the
other notebooks.